In [5]:
import tskit
import pandas as pd
from lets_plot import *
LetsPlot.setup_html()
import numpy as np
from collections import Counter

chromosome = 15

In [25]:
# See how many are flipped
flipped = {}
for num in range(99):
    muts = pd.read_table("Chr" + str(chromosome) + "_NonH_Chunk" + chunk + "/Chr" + str(chromosome) + "_NonHybrid_" + chunk + "_muts_" + str(num) + ".txt", header=None, sep=" ", names=["Pos", "X", "ParentNode", "Der"])
    flipped[num] = len(muts[(muts.ParentNode == -1) & (muts.Der == 1)])

In [26]:
flippedDF = pd.DataFrame.from_dict(flipped, orient = "index").reset_index()
flippedDF.columns = ["Num", "Flipped"]

ggplot(data = flippedDF) + geom_line(aes(x = "Num", y = "Flipped"))

In [ ]:
diversity = {}
for num in range(1, 100):
    ts = tskit.load("/sietch_colab/jobste/Singer/Chr" + str(chromosome) + "/Chr" + str(chromosome) + "_NonHybrid_" + str(num) + ".trees")
    diversity[num] = ts.diversity(mode = "site") 

In [12]:
diversityDF = pd.DataFrame.from_dict(diversity, orient = "index")
diversityDF.loc[:, "Num"] = diversityDF.index
diversityDF.columns = ["Diversity", "Num"]
diversityDF.loc[:, "Diversity"] = diversityDF.Diversity.astype(float)

ggplot(data = diversityDF) + geom_line(aes(x = "Num", y = "Diversity"))

In [13]:

ts = tskit.load("/sietch_colab/jobste/Singer/Chr" + str(chromosome) + "/Chr" + str(chromosome) + "_NonHybrid_99.trees")
windowsList = np.linspace(0, ts.sequence_length, num = 1000)
windowDiversity = ts.diversity(windows=windowsList)
diversityW = pd.DataFrame({"Breakpoint": windowsList[:-1], "Diversity": list(windowDiversity)})



In [5]:
ggplot(data = diversityW) + geom_line(aes(x = "Breakpoint", y = "Diversity"))

In [6]:
np.mean(list(diversity.values())[-10:-1])

0.0035073103601568085

In [7]:
# Compare to the latest Relate trees
remappedDiv = pd.read_csv("../Relate/Remapped/WindowedDiversity_" + str(chromosome) + ".csv")

In [9]:
np.mean(remappedDiv.Diversity)

0.003011676622224141

In [10]:
ggplot(data = remappedDiv) + geom_line(aes(x = "Breakpoint", y = "Diversity")) + facet_wrap(facets = "Pop")

In [11]:
rmap = pd.read_table("/home/jobste/Data/FilteringData/Chr16_remapped_A.rmap", header=None, names=["Start", "Stop", "R"])


In [17]:
ggplot(data = rmap[rmap.Start > 200000]) + geom_line(aes(x = "Start", y = "R"))